In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

import os
def get_sqlalchemy_engine():
    host = os.getenv("DB_HOST")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")
    dbname = "animedw"
    port = int(os.getenv("DB_PORT",5432))
    conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
    engine = create_engine(conn_str)
    return engine

def map_metadata_to_anime(df_raw : pd.DataFrame, prefix : str) -> pd.DataFrame:
    df_working = df_raw[['anime_mal_id', prefix]].copy()
    try:
        
        df_working = df_working[["anime_mal_id",prefix]].explode(prefix)
        df_working = df_working.dropna(subset = [prefix])
        df_flat = pd.json_normalize(df_working[prefix])

        df_working.reset_index(drop=True, inplace= True)
        df_flat.reset_index(drop=True, inplace= True)

        prefix = prefix.rstrip("s")
        
        if prefix in ["producer", "studio", "licensor"]:

            df_flat = df_flat.rename(columns={"mal_id" : "organization_mal_id"}).assign(role = prefix)
            concat_list = [df_working["anime_mal_id"],df_flat[["organization_mal_id","role"]]]
            return pd.concat(concat_list, axis = 1)

        else:

            df_flat = df_flat.rename(columns={"mal_id" : f"{prefix}_mal_id"})
            concat_list = [df_working["anime_mal_id"],df_flat[f"{prefix}_mal_id"]]
            return pd.concat(concat_list, axis = 1)

    except Exception as e:
        print(f"Error Occurelí {e}")


def map_organization_metadata_to_anime(df_raw : pd.DataFrame) -> pd.DataFrame:
    try:
        df_anime_producer = map_metadata_to_anime(df_raw, "producers")
        df_anime_studio = map_metadata_to_anime(df_raw, "studios")
        df_anime_licensor = map_metadata_to_anime(df_raw, "licensors")
        concat_list = [df_anime_licensor,df_anime_producer,df_anime_studio]
        return pd.concat(concat_list)
    
    except Exception as e:
        print(f"Failed mapping organization metadata to anime: {e}")

In [32]:
engine = get_sqlalchemy_engine() 

query = "SELECT payload FROM bronze.anime_raw"
payload = pd.read_sql(query,engine)
# flatten json into fields
df_raw = pd.json_normalize(payload["payload"])
df_raw.rename(columns = {"mal_id":"anime_mal_id"}, inplace = True)

def process_metadata(df_raw : pd.DataFrame, prefix : str):
    df_working = df_raw[prefix].explode() 
    df_working = df_working.dropna()
    df_working = pd.json_normalize(df_working)
    df_working = df_working[["mal_id","name","url"]].drop_duplicates()
    return df_working.sort_values("mal_id")

In [39]:
df = process_metadata(df_raw, "producers")
df

,mal_id,name,url
13,1,Studio Pierrot,https://myanimelist.net/anime/producer/1/Studi...
11619,3,Gonzo,https://myanimelist.net/anime/producer/3/Gonzo
2729,4,Bones,https://myanimelist.net/anime/producer/4/Bones
13062,5,Bee Train,https://myanimelist.net/anime/producer/5/Bee_T...
11309,6,Gainax,https://myanimelist.net/anime/producer/6/Gainax
...,...,...,...
2365,3285,Mango TV,https://myanimelist.net/anime/producer/3285/Ma...
11206,3289,TIMOTA,https://myanimelist.net/anime/producer/3289/TI...
14723,3291,Sai Enterprises,https://myanimelist.net/anime/producer/3291/Sa...
14635,3292,Studio Tamatsubaki,https://myanimelist.net/anime/producer/3292/St...
